# Занятие 2. Практика: NumPy, скорость и первый анализ данных — АВТОРСКОЕ РЕШЕНИЕ

На прошлом занятии мы познакомились с ноутбуком, NumPy и Pandas.
Сегодня:
1. сравним скорость NumPy и обычных списков на миллионе чисел;
2. посчитаем агрегаты без циклов;
3. прочитаем настоящие данные (CSV и Excel);
4. сделаем первичный осмотр таблицы (`.info()`, `.describe()`);
5. научимся выбирать строки и столбцы через `.loc` и `.iloc`.

Датасет — подготовка **1000 школьников к ЕГЭ**.


In [2]:
# импортируем библиотеки
import numpy as np
import pandas as pd

---
## Часть 1. NumPy против списков: кто быстрее?

Возьмём **миллион** чисел и сложим их двумя способами: обычным списком и массивом NumPy.
Замерим время волшебной командой `%timeit` — она сама несколько раз запускает код
и показывает среднее время.


In [3]:
N = 1_000_000
py_list = list(range(N))       # обычный список
np_array = np.arange(N)        # массив NumPy
print('Готово. В каждом по', N, 'чисел.')

Готово. В каждом по 1000000 чисел.


### Что такое `%timeit`

`%timeit` — это «секундомер» ноутбука. Он берёт вашу строку кода и запускает её **много раз**,
а потом показывает, **сколько в среднем** времени она занимает. Так измерение получается точным,
а не случайным.

Пишется просто: `%timeit` и дальше код, время которого измеряем. Например, `%timeit sum(py_list)`.

**Как читать результат.** После запуска появится строка примерно такого вида:

```
156 µs ± 3.18 µs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
```

Разберём её по частям:

| Кусок | Что значит простыми словами |
|---|---|
| `156 µs` | **среднее время одного запуска** — 156 микросекунд |
| `± 3.18 µs` | насколько результат «прыгает» от запуска к запуску (разброс) |
| `per loop` | «за один прогон» кода |
| `7 runs` | измерение повторили 7 раз, чтобы быть точнее |
| `10,000 loops each` | в каждом измерении код прогнали 10 000 раз |

> **µs — это микросекунда**, одна миллионная доля секунды (0.000001 с).
> Бывают ещё `ms` — миллисекунды (тысячная секунды) и `s` — секунды.
> Порядок по возрастанию: **µs → ms → s**.

**Главное правило:** чем **меньше** число перед `µs`/`ms`/`s`, тем **быстрее** код.
Дальше мы сравним время у списка и у массива NumPy — и увидим разницу своими глазами.

### Сумма чисел


In [4]:
# Способ 1: сумма списка через встроенный sum()
%timeit sum(py_list)

6.32 ms ± 38.8 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [5]:
# Способ 2: сумма массива через NumPy
%timeit np_array.sum()

142 µs ± 4.18 µs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


Сравните числа: у NumPy время в **десятки раз меньше**. Причина — векторизация:
NumPy складывает всё сразу, а не по одному элементу в цикле.

### Умножение каждого числа на 2


In [6]:
# Список: нужен цикл (списковое включение)
%timeit [x * 2 for x in py_list]

40.1 ms ± 1 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [7]:
# Массив: одна операция сразу над всеми
%timeit np_array * 2

811 µs ± 6.72 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


**Вывод:** для вычислений с числами массив NumPy намного быстрее и короче в записи.
Именно поэтому Pandas построен на NumPy.


---
## Часть 2. Агрегаты без циклов

На массиве легко считать статистику одной командой.


In [8]:
data = np.array([90, 75, 60, 82, 95, 70, 88, 64])

print('сумма   :', np.sum(data))
print('среднее :', np.mean(data))
print('минимум :', np.min(data))
print('максимум:', np.max(data))

сумма   : 624
среднее : 78.0
минимум : 60
максимум: 95


Эти же функции работают и на больших массивах мгновенно.


In [9]:
big_array = np.arange(1, N + 1)     # числа от 1 до миллиона
print('сумма миллиона чисел:', big_array.sum())
print('среднее:', big_array.mean())

сумма миллиона чисел: 500000500000
среднее: 500000.5


---
## Часть 3. Читаем настоящие данные

Данные обычно лежат в файлах **CSV** или **Excel (XLSX)**. Pandas читает оба формата.

### Что такое CSV

**CSV** (*Comma-Separated Values* — «значения, разделённые запятыми») — это обычный
**текстовый файл**, где таблица записана строками, а ячейки в строке разделены
каким-то символом-разделителем (запятой `,` или точкой с запятой `;`). Первая строка
обычно содержит названия столбцов. Открыть такой файл можно даже в «Блокноте» —
внутри просто текст:

```
имя;город;балл_егэ
Роман;Нижний Новгород;47
Арина;Ростов-на-Дону;66
```

### Чем CSV лучше XLSX

XLSX — это формат Excel: сложный архив со стилями, формулами, вкладками и оформлением.
Для хранения **данных** CSV чаще удобнее:

| | CSV | XLSX (Excel) |
|---|---|---|
| **Что внутри** | простой текст | сложный архив (стили, формулы, вкладки) |
| **Размер** | маленький (только данные) | больше (тащит с собой оформление) |
| **Скорость чтения** | быстро | медленнее |
| **Кто откроет** | любая программа, любой язык | нужен Excel или спец. библиотека |
| **Годится для программ** | да, это стандарт для обмена данными | не всегда |

Коротко: **CSV — универсальный, лёгкий и быстрый** формат для данных, поэтому его
используют почти везде. XLSX удобен, когда нужны формулы, графики и красивое оформление
внутри самого файла, но как «хранилище чисел» он тяжелее и медленнее.

### Чем читаем таблицы: метод `read_...`

Для чтения таблиц в Pandas используют семейство методов **`read_...`**:
`pd.read_csv(...)` — для CSV, `pd.read_excel(...)` — для Excel. Достаточно указать
путь к файлу — и таблица загрузится в `DataFrame`.

У этих методов **много параметров**, которые управляют чтением. Например, `sep`
задаёт символ-разделитель в CSV (у нас `sep=';'`). Есть и другие: сколько строк
пропустить, какую строку считать заголовком, какие столбцы взять и так далее.
Пока запомним только то, что нужно сейчас, — **с остальными параметрами мы
подробнее познакомимся в будущем**.


In [10]:
# CSV: разделитель в нашем файле — точка с запятой (sep=';')
df = pd.read_csv('data/ege_students.csv', sep=';')
df.head()

,ученик_id,имя,пол,возраст,город,тип_школы,любимый_предмет,часов_подготовки_в_неделю,репетитор,кол_во_пробников,пропусков_уроков,мотивация_1_10,часов_сна,средний_балл_года,балл_егэ,сдал
0,1,Роман,М,17,Нижний Новгород,Обычная,Физика,5.4,нет,7,11,5.0,7.9,2.4,47,да
1,2,Арина,Ж,17,Ростов-на-Дону,Обычная,Математика,9.1,нет,8,2,2.0,7.9,3.4,66,да
2,3,Дарья,Ж,17,Самара,Обычная,Русский язык,11.3,да,14,4,7.0,8.2,2.6,80,да
3,4,Владимир,М,16,Ростов-на-Дону,Обычная,Химия,6.2,да,4,5,1.0,5.0,2.6,58,да
4,5,Дмитрий,М,17,Москва,Обычная,Русский язык,6.5,нет,6,4,2.0,6.8,3.0,61,да


In [11]:
# То же самое можно прочитать из Excel:
df_excel = pd.read_excel('data/ege_students.xlsx')
df_excel.head()

,ученик_id,имя,пол,возраст,город,тип_школы,любимый_предмет,часов_подготовки_в_неделю,репетитор,кол_во_пробников,пропусков_уроков,мотивация_1_10,часов_сна,средний_балл_года,балл_егэ,сдал
0,1,Роман,М,17,Нижний Новгород,Обычная,Физика,5.4,нет,7,11,5.0,7.9,2.4,47,да
1,2,Арина,Ж,17,Ростов-на-Дону,Обычная,Математика,9.1,нет,8,2,2.0,7.9,3.4,66,да
2,3,Дарья,Ж,17,Самара,Обычная,Русский язык,11.3,да,14,4,7.0,8.2,2.6,80,да
3,4,Владимир,М,16,Ростов-на-Дону,Обычная,Химия,6.2,да,4,5,1.0,5.0,2.6,58,да
4,5,Дмитрий,М,17,Москва,Обычная,Русский язык,6.5,нет,6,4,2.0,6.8,3.0,61,да


---
## Часть 4. Первичный осмотр таблицы

Прежде чем что-то анализировать, таблицу всегда **осматривают**. Три главных инструмента:
- **`.head()`** — первые строки (по умолчанию 5);
- **`.info()`** — сводка: столбцы, типы, пропуски;
- **`.describe()`** — статистика по числовым столбцам.


In [12]:
# Первые 5 строк
df.head()

,ученик_id,имя,пол,возраст,город,тип_школы,любимый_предмет,часов_подготовки_в_неделю,репетитор,кол_во_пробников,пропусков_уроков,мотивация_1_10,часов_сна,средний_балл_года,балл_егэ,сдал
0,1,Роман,М,17,Нижний Новгород,Обычная,Физика,5.4,нет,7,11,5.0,7.9,2.4,47,да
1,2,Арина,Ж,17,Ростов-на-Дону,Обычная,Математика,9.1,нет,8,2,2.0,7.9,3.4,66,да
2,3,Дарья,Ж,17,Самара,Обычная,Русский язык,11.3,да,14,4,7.0,8.2,2.6,80,да
3,4,Владимир,М,16,Ростов-на-Дону,Обычная,Химия,6.2,да,4,5,1.0,5.0,2.6,58,да
4,5,Дмитрий,М,17,Москва,Обычная,Русский язык,6.5,нет,6,4,2.0,6.8,3.0,61,да


In [13]:
# Сводка: сколько строк, какие типы, где пропуски
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 16 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   ученик_id                  1000 non-null   int64  
 1   имя                        1000 non-null   object 
 2   пол                        1000 non-null   object 
 3   возраст                    1000 non-null   int64  
 4   город                      1000 non-null   object 
 5   тип_школы                  1000 non-null   object 
 6   любимый_предмет            1000 non-null   object 
 7   часов_подготовки_в_неделю  1000 non-null   float64
 8   репетитор                  1000 non-null   object 
 9   кол_во_пробников           1000 non-null   int64  
 10  пропусков_уроков           1000 non-null   int64  
 11  мотивация_1_10             975 non-null    float64
 12  часов_сна                  960 non-null    float64
 13  средний_балл_года          1000 non-null   float6

В выводе `.info()` смотрите на два момента:
- **Non-Null Count** — сколько заполненных значений в столбце;
- **Dtype** — тип столбца (число `int/float` или текст `object`).


In [14]:
# Статистика по числовым столбцам: среднее, минимум, максимум и др.
df.describe().round(1)

,ученик_id,возраст,часов_подготовки_в_неделю,кол_во_пробников,пропусков_уроков,мотивация_1_10,часов_сна,средний_балл_года,балл_егэ
count,1000.0,1000.0,1000.0,1000.0,1000.0,975.0,960.0,1000.0,1000.0
mean,500.5,17.0,8.0,6.4,5.7,6.1,7.0,3.0,67.1
std,288.8,0.6,4.8,2.7,3.1,2.2,1.2,0.7,18.2
min,1.0,16.0,0.0,0.0,0.0,1.0,4.0,2.0,20.0
25%,250.8,17.0,4.8,4.8,4.0,5.0,6.2,2.5,54.0
50%,500.5,17.0,8.0,6.0,6.0,6.0,7.0,3.0,68.0
75%,750.2,17.0,10.9,8.0,8.0,8.0,7.8,3.5,81.0
max,1000.0,18.0,26.8,16.0,16.0,10.0,10.8,5.0,100.0


`.describe()` сразу отвечает на вопросы: какой средний балл ЕГЭ, минимальный и максимальный,
сколько в среднем часов готовятся ученики и т. д.


In [15]:
# Отдельно посчитаем главные агрегаты по баллу ЕГЭ
print('Средний балл ЕГЭ :', df['балл_егэ'].mean().round(1))
print('Максимальный балл:', df['балл_егэ'].max())
print('Минимальный балл :', df['балл_егэ'].min())

Средний балл ЕГЭ : 67.1
Максимальный балл: 100
Минимальный балл : 20


---
## Часть 5. Индексация: как достать нужные строки и столбцы

Таблица `df` — это как большая таблица в Excel. Часто нам нужна не вся она,
а только кусочек: один столбец, одна строка, несколько строк подряд или
конкретная «клетка». Разберём все способы по порядку.

### 5.1. Обращение к столбцу через квадратные скобки

Самый частый приём — взять **один столбец по имени** в квадратных скобках `df['имя_столбца']`.
Получится «столбик» значений (объект `Series`).


In [16]:
# один столбец по названию
df['балл_егэ'].head()

0    47
1    66
2    80
3    58
4    61
Name: балл_егэ, dtype: int64

### 5.2. Обращение к столбцу через точку

Если в названии столбца **нет пробелов и спецсимволов**, к нему можно обратиться
короче — через точку: `df.столбец`. Это то же самое, что и квадратные скобки.

> ⚠️ Через точку **нельзя**, если в имени есть пробел (например, `df.любимый предмет`
> не сработает — тут только `df['любимый предмет']`). Квадратные скобки работают всегда.


In [17]:
# то же самое через точку (пробелов в имени нет — можно)
df.балл_егэ.head()

0    47
1    66
2    80
3    58
4    61
Name: балл_егэ, dtype: int64

### 5.3. Несколько столбцов сразу

Чтобы взять **несколько столбцов**, передаём **список имён** — обратите внимание на
**двойные скобки** `[[ ... ]]`: внешние — это индексация, внутренние — список.


In [18]:
# несколько столбцов: список имён в двойных скобках
df[['имя', 'город', 'балл_егэ']].head()

,имя,город,балл_егэ
0,Роман,Нижний Новгород,47
1,Арина,Ростов-на-Дону,66
2,Дарья,Самара,80
3,Владимир,Ростов-на-Дону,58
4,Дмитрий,Москва,61


### 5.4. Два инструмента для строк и клеток: `.loc` и `.iloc`

Когда нужны **строки** (или конкретные клетки), используют два инструмента:

| Инструмент | Как обращаемся | Пример |
|---|---|---|
| **`.loc`** | по **названиям** (метка строки, имя столбца) | `df.loc[0, 'имя']` |
| **`.iloc`** | по **номерам-позициям** (счёт с 0) | `df.iloc[0, 1]` |

Формат у обоих одинаковый: `df.loc[строки, столбцы]` и `df.iloc[строки, столбцы]`.
У нашей таблицы метки строк — это числа `0, 1, 2, ...`, поэтому для строк
`.loc` и `.iloc` часто выглядят похоже, но это **разные вещи** (см. про срезы ниже).

### 5.5. Одна конкретная строка

`.loc[метка]` — строка по её метке, `.iloc[номер]` — строка по её позиции.


In [19]:
# строка с меткой 0 (первый ученик)
df.loc[0]

ученик_id                                  1
имя                                    Роман
пол                                        М
возраст                                   17
город                        Нижний Новгород
тип_школы                            Обычная
любимый_предмет                       Физика
часов_подготовки_в_неделю                5.4
репетитор                                нет
кол_во_пробников                           7
пропусков_уроков                          11
мотивация_1_10                           5.0
часов_сна                                7.9
средний_балл_года                        2.4
балл_егэ                                  47
сдал                                      да
Name: 0, dtype: object

In [20]:
# та же строка по позиции (нумерация с нуля)
df.iloc[0]

ученик_id                                  1
имя                                    Роман
пол                                        М
возраст                                   17
город                        Нижний Новгород
тип_школы                            Обычная
любимый_предмет                       Физика
часов_подготовки_в_неделю                5.4
репетитор                                нет
кол_во_пробников                           7
пропусков_уроков                          11
мотивация_1_10                           5.0
часов_сна                                7.9
средний_балл_года                        2.4
балл_егэ                                  47
сдал                                      да
Name: 0, dtype: object

### 5.6. Одна конкретная клетка (строка + столбец)

Указываем и строку, и столбец. Через `.loc` — по именам, через `.iloc` — по номерам.


In [21]:
# клетка: строка 0, столбец 'имя'
print('По названию :', df.loc[0, 'имя'])

# та же клетка по номерам: строка 0, столбец 1 ('имя' — второй столбец, счёт с 0)
print('По номеру   :', df.iloc[0, 1])

По названию : Роман
По номеру   : Роман


### 5.7. Один столбец через `.loc` / `.iloc`

Если поставить `:` на месте строк — значит «все строки».


In [22]:
# все строки, столбец 'город' — первые 5
df.loc[:, 'город'].head()

0    Нижний Новгород
1     Ростов-на-Дону
2             Самара
3     Ростов-на-Дону
4             Москва
Name: город, dtype: object

In [23]:
# все строки, столбец с номером 4 ('город') — первые 5
df.iloc[:, 4].head()

0    Нижний Новгород
1     Ростов-на-Дону
2             Самара
3     Ростов-на-Дону
4             Москва
Name: город, dtype: object

### 5.8. Срезы: несколько строк подряд

Срез — это диапазон «от и до». Здесь у `.loc` и `.iloc` есть **важное отличие**:

- **`.loc[3:6]`** — по меткам, **правая граница ВКЛЮЧАЕТСЯ** → строки 3, 4, 5, **6**;
- **`.iloc[3:6]`** — по позициям, как в обычном Python, **правая граница НЕ включается**
  → строки 3, 4, 5 (без 6).

Запомнить легко: `.loc` — «по-человечески, до и включая», `.iloc` — «как в Python, до, но не включая».


In [24]:
# .loc[3:6] — включает строку 6 → всего 4 строки
df.loc[3:6, ['имя', 'балл_егэ']]

,имя,балл_егэ
3,Владимир,58
4,Дмитрий,61
5,Милана,81
6,Роман,90


In [25]:
# .iloc[3:6] — НЕ включает строку 6 → всего 3 строки
df.iloc[3:6, [1, 14]]

,имя,балл_егэ
3,Владимир,58
4,Дмитрий,61
5,Милана,81


### 5.9. Срез от начала и до конца

Границу можно не писать: пусто слева — «с самого начала», пусто справа — «до самого конца».


In [26]:
# первые 5 строк: от начала до позиции 5 (не включая)
df.iloc[:5]

,ученик_id,имя,пол,возраст,город,тип_школы,любимый_предмет,часов_подготовки_в_неделю,репетитор,кол_во_пробников,пропусков_уроков,мотивация_1_10,часов_сна,средний_балл_года,балл_егэ,сдал
0,1,Роман,М,17,Нижний Новгород,Обычная,Физика,5.4,нет,7,11,5.0,7.9,2.4,47,да
1,2,Арина,Ж,17,Ростов-на-Дону,Обычная,Математика,9.1,нет,8,2,2.0,7.9,3.4,66,да
2,3,Дарья,Ж,17,Самара,Обычная,Русский язык,11.3,да,14,4,7.0,8.2,2.6,80,да
3,4,Владимир,М,16,Ростов-на-Дону,Обычная,Химия,6.2,да,4,5,1.0,5.0,2.6,58,да
4,5,Дмитрий,М,17,Москва,Обычная,Русский язык,6.5,нет,6,4,2.0,6.8,3.0,61,да


In [27]:
# последние 5 строк: с позиции -5 и до конца
df.iloc[-5:]

,ученик_id,имя,пол,возраст,город,тип_школы,любимый_предмет,часов_подготовки_в_неделю,репетитор,кол_во_пробников,пропусков_уроков,мотивация_1_10,часов_сна,средний_балл_года,балл_егэ,сдал
995,996,София,Ж,17,Москва,Обычная,История,8.4,нет,5,6,2.0,7.1,2.6,59,да
996,997,Дарья,Ж,17,Казань,Гимназия,Химия,4.3,да,1,4,5.0,NaN,2.9,66,да
997,998,Матвей,М,17,Самара,Обычная,История,5.9,нет,4,7,2.0,6.9,2.4,48,да
998,999,Алиса,Ж,18,Ростов-на-Дону,Обычная,Русский язык,3.1,нет,9,6,8.0,6.2,3.9,74,да
999,1000,Алиса,Ж,17,Екатеринбург,Обычная,Физика,14.4,да,5,3,8.0,6.2,4.2,89,да


In [28]:
# все строки, но только столбцы с 1-го по 4-й (по номерам)
df.iloc[:, 1:5].head()

,имя,пол,возраст,город
0,Роман,М,17,Нижний Новгород
1,Арина,Ж,17,Ростов-на-Дону
2,Дарья,Ж,17,Самара
3,Владимир,М,16,Ростов-на-Дону
4,Дмитрий,М,17,Москва


---
## Практика — авторское решение

Ниже — 12 заданий на изученный материал. В каждом задании:
- **напишите код** в ячейке `# Ваш код` — код обязателен;
- **выведите результат** через `print()` или последней строкой.

В некоторых заданиях есть пункт **Вывод:** — там нужно короткое предложение о том, что получилось и как это трактовать.

Работаем с таблицей `df`, загруженной выше. Столбцы:
`ученик_id`, `имя`, `пол`, `возраст`, `город`, `тип_школы`,
`любимый_предмет`, `часов_подготовки_в_неделю`, `репетитор`, `кол_во_пробников`,
`пропусков_уроков`, `мотивация_1_10`, `часов_сна`, `средний_балл_года`, `балл_егэ`, `сдал`.

**Максимум за практику — 30 баллов.**


### <font color='DarkOrange'>Задание 1 [1 балл]</font>

Определите размер таблицы `df` — сколько в ней **строк** и **столбцов**.


In [29]:
df.shape
# Ответ: (1000, 16) — 1000 строк, 16 столбцов

(1000, 16)

### <font color='DarkOrange'>Задание 2 [2 балла]</font>

Найдите **имя ученика** в строке с меткой **500**. Используйте `.loc`.


In [30]:
df.loc[500, 'имя']
# Ответ: Михаил

'Михаил'

### <font color='DarkOrange'>Задание 3 [2 балла]</font>

Найдите **максимальный** и **минимальный** балл ЕГЭ в таблице.


In [31]:
print('Максимум:', df['балл_егэ'].max())
print('Минимум :', df['балл_егэ'].min())
# Ответ: максимум 100, минимум 20

Максимум: 100
Минимум : 20


**Вывод:** разрыв в 80 баллов показывает, что подготовка учеников очень разная — от совсем слабой до отличной.


### <font color='DarkOrange'>Задание 4 [2 балла]</font>

Посчитайте **средний балл ЕГЭ** по всем ученикам и **округлите до одного знака** после запятой.


In [32]:
round(df['балл_егэ'].mean(), 1)
# Ответ: 67.1

67.1

### <font color='DarkOrange'>Задание 5 [2 балла]</font>

Какой **любимый предмет** у ученика в строке с меткой **100**? Используйте `.loc`.


In [33]:
df.loc[100, 'любимый_предмет']
# Ответ: Физика

'Физика'

### <font color='DarkOrange'>Задание 6 [3 балла]</font>

Из какого **города** первый ученик таблицы (строка с позицией 0)?
Достаньте значение через **`.iloc`** — по номерам строки и столбца.

Столбец `город` — пятый по счёту, то есть индекс 4 (счёт с нуля).


In [34]:
df.iloc[0, 4]
# Ответ: Нижний Новгород

'Нижний Новгород'

### <font color='DarkOrange'>Задание 7 [2 балла]</font>

Сколько часов в неделю готовился ученик с **самой долгой подготовкой**?


In [35]:
df['часов_подготовки_в_неделю'].max()
# Ответ: 26.8

26.8

### <font color='DarkOrange'>Задание 8 [3 балла]</font>

Возьмите **последние 100 учеников** таблицы и посчитайте у них **средний балл ЕГЭ**
(округлите до одного знака после запятой).


In [36]:
round(df.iloc[-100:]['балл_егэ'].mean(), 1)
# Ответ: 63.4

63.4

### <font color='DarkOrange'>Задание 9 [3 балла]</font>

Среди **первых 250 учеников** найдите **максимальный возраст**.


In [37]:
df.iloc[:250]['возраст'].max()
# Ответ: 18

18

### <font color='DarkOrange'>Задание 10 [3 балла]</font>

Возьмите учеников **с позиции 300 по 399** (сто человек) и посчитайте их
**среднее число часов сна** (округлите до одного знака после запятой).


In [38]:
round(df.iloc[300:400]['часов_сна'].mean(), 1)
# Ответ: 7.1

7.1

### <font color='DarkOrange'>Задание 11 [3 балла]</font>

Возьмите учеников **с позиции 100 по 249** (150 человек) и посчитайте их
**среднее количество пробников** (столбец `кол_во_пробников`, округлите до одного знака после запятой).


In [39]:
round(df.iloc[100:250]['кол_во_пробников'].mean(), 1)
# Ответ: 6.2

6.2

### <font color='DarkOrange'>Задание 12 [4 балла]</font>

Возьмите строки **с меткой 500 по 600 включительно** (через `.loc`) и посчитайте
их **средний балл за год** (столбец `средний_балл_года`, округлите до двух знаков).


In [40]:
round(df.loc[500:600, 'средний_балл_года'].mean(), 2)
# Ответ: 2.98

2.98

**Вывод:** в срез `.loc[500:600]` попадает **101 строка** (правая граница включается), их средний балл за год — **2.98**.
